# Robustness and Stability Analysis

This notebook evaluates the **robustness and stability** of our primary $K=3$ VMPFC parcellation. To ensure that the identified functional subdivisions are not mere artifacts of a specific algorithm, initialization bias, or noise threshold, we systematically replicate the parcellation under various strict conditions.

Specifically, we test alternative clustering algorithms (Spectral and Agglomerative) and varying data inclusion thresholds across hundreds of random iterations to confirm the reproducibility of the topographical boundaries.

In [1]:
from neurosynth import Dataset
from neurosynth.analysis.cluster import Clusterable
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances, davies_bouldin_score, silhouette_score
from sklearn.cluster import SpectralClustering
from sklearn.decomposition import PCA
from copy import deepcopy
import joblib
from pathlib import Path
import nibabel as nib
import numpy as np
import nilearn.plotting as nplt
from tqdm import trange
import matplotlib.pyplot as plt
import warnings

plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'Arial'
warnings.filterwarnings("ignore", category=FutureWarning)

# Path Settings
DATA_PATH = Path('../data')
RESULTS_PATH = Path('../results')
DATASET_FILE = DATA_PATH / 'neurosynth_data/dataset.pkl'
ROI_FILE = DATA_PATH / 'masks/VMPFC_mask_v2.nii'
unROI_FILE = DATA_PATH / 'masks/unVMPFC_mask_v2.nii'
N_CLUSTER_LIST = list(range(2, 7))
MIN_STUDIES = 150 # min studies for activation
N_COMPONENTS = 100 # number of PCA components

# load mask image
# ROI mask and co-activation region (whole brain or region after removing ROI)
ROI_IMAGE = nib.load(str(ROI_FILE))
print(f'ROI mask includes {ROI_IMAGE.get_fdata().sum():.0f} voxels.')
unROI_IMAGE = nib.load(str(unROI_FILE))
print(f'unROI mask includes {unROI_IMAGE.get_fdata().sum():.0f} voxels.')

/home/guoqiu/.conda/envs/vmpfc_nsplus/lib/python3.8/site-packages/sklearn/utils/deprecation.py:143: FutureWarning: The sklearn.feature_selection.univariate_selection module is  deprecated in version 0.22 and will be removed in version 0.24. The corresponding classes / functions should instead be imported from sklearn.feature_selection. Anything that cannot be imported from sklearn.feature_selection is now part of the private API.
  warnings.warn(message, FutureWarning)


# Prepare data
NeuroSynth's data includes **activation maps** that represent patterns of brain activity linked to specific cognitive or psychological terms.

In [4]:
dataset = Dataset.load(DATASET_FILE)
print(f'Dataset includes {dataset.feature_table.data.shape[0]} studies.')

Dataset includes 14371 studies.


In [5]:
# extract masked dataset (ROI and unROI), which is activation data
ROI_dataset = Clusterable(dataset, mask=ROI_IMAGE)
ROI_activation_raw = ROI_dataset.data
print(f'ROI dataset: (n voxels, n studies) = {ROI_dataset.data.shape}.')

# masker, to map vector back to brain voxels
ROI_masker = deepcopy(dataset.masker)
ROI_masker.add({'roi': ROI_IMAGE})

# remaining voxels to compare
unROI_dataset = Clusterable(dataset, mask=unROI_IMAGE)
unROI_activation_raw = unROI_dataset.data
print(f'unROI dataset: (n voxels, n studies) = {unROI_dataset.data.shape}.')

# masker, to map vector back to brain voxels
unROI_masker = deepcopy(dataset.masker)
unROI_masker.add({'roi': unROI_IMAGE})

ROI dataset: (n voxels, n studies) = (3697, 14371).
unROI dataset: (n voxels, n studies) = (224756, 14371).


# Co-activation matrix
For further analysis, coactivation maps were used, which are somewhat similar to functional connectivity but fundamentally different. While functional connectivity measures correlations in brain activity over time, coactivation maps reflect patterns of brain regions that consistently activate together across different studies, providing insights into shared functional networks.

One major consideration was the minimum number of studies reporting activation, as this affects which voxels are included in further analysis.

In [7]:
# map of voxels changes based on the minimum number of studies
ROI_studies_per_voxel = ROI_activation_raw.sum(1)
ROI_image = ROI_masker.unmask(ROI_studies_per_voxel)
ROI_image = nib.nifti1.Nifti1Image(ROI_image, affine=ROI_IMAGE.affine, )

unROI_studies_per_voxel = unROI_activation_raw.sum(1)
unROI_image = unROI_masker.unmask(unROI_studies_per_voxel)
unROI_image = nib.nifti1.Nifti1Image(unROI_image, affine=unROI_IMAGE.affine, )

ROI_voxels_to_use = ROI_studies_per_voxel >= MIN_STUDIES
ROI_activation = ROI_activation_raw[ROI_voxels_to_use, :]
print(f'ROI dataset: (n voxels, n studies) = {ROI_activation.data.shape}.')

unROI_voxels_to_use = unROI_studies_per_voxel >= MIN_STUDIES
unROI_activation = unROI_activation_raw[unROI_voxels_to_use, :]
print(f'unROI dataset: (n voxels, n studies) = {unROI_activation.data.shape}.')

ROI dataset: (n voxels, n studies) = (1784, 14371).
unROI dataset: (n voxels, n studies) = (129213, 14371).


# Spectral Clustering

To verify that our results are independent of the geometric assumptions made by K-Means (which tends to favor spherical clusters), we apply **Spectral Clustering**. This graph-based approach groups voxels based on the eigenvectors of the co-activation distance matrix, allowing for more complex, non-linear cluster boundaries. We compute the $K=3$ solution across 100 random initializations to assess boundary stability.

In [8]:
for random_state in trange(100):
    this_result_path = RESULTS_PATH / 'robust/runs_Spectral'
    this_result_path.mkdir(parents=True, exist_ok=True)

    # decompose the data to remove noise and reduce data dimensions
    pca = PCA(n_components=N_COMPONENTS, svd_solver='randomized', random_state=random_state)
    unROI_activation_PCs = pca.fit_transform(unROI_activation.T).T

    # Finding the Pearson distance (1 - r) between voxels of ROI and unROI PCs
    coactivation = pairwise_distances(ROI_activation, unROI_activation_PCs, metric='correlation', n_jobs=64)
    label_dict = {
        n_clusters:
            SpectralClustering(
                n_clusters=n_clusters,
                n_neighbors=20,
                assign_labels='kmeans',
                random_state=random_state,
                n_jobs=-1
            ).fit_predict(coactivation) + 1
        for n_clusters in [3]
    }

    # plot and save image
    for cluster_n in [3]:
        label = np.zeros_like(ROI_studies_per_voxel, dtype=np.int8)
        label[ROI_voxels_to_use] = label_dict[cluster_n]
        label_img_data = ROI_masker.unmask(label)
        label_img = nib.nifti1.Nifti1Image(label_img_data, header=ROI_IMAGE.header, affine=ROI_IMAGE.affine, )
        nib.save(label_img, str(this_result_path / f'run_{random_state}_K{cluster_n}.nii.gz'))


100%|██████████| 100/100 [2:05:59<00:00, 75.59s/it] 


# Agglomerative Clustering

Next, we validate the parcellation using **Agglomerative (Hierarchical) Clustering**. Crucially, we introduce a strict spatial constraint (`spatial_connectivity`) generated directly from the 3D grid of the VMPFC mask. By enforcing this connectivity, the algorithm is forced to only merge voxels that are anatomically contiguous. Using Ward's linkage, we run another 100 iterations to ensure our functional subregions remain cohesive and are not driven by spatially disconnected noise.

In [ ]:
ROI_mask_3d = ROI_masker.unmask(ROI_voxels_to_use).astype(bool)
spatial_connectivity = grid_to_graph(*ROI_mask_3d.shape, mask=ROI_mask_3d)
n_components, labels = connected_components(csgraph=spatial_connectivity, directed=False, return_labels=True)
print(f'We have {n_components} islands')
unique_islands, voxel_counts = np.unique(labels, return_counts=True)
for island_id, count in zip(unique_islands, voxel_counts):
    print(f"island {island_id + 1} has {count} voxels")

for random_state in trange(100):
    this_result_path = RESULTS_PATH / 'robust/runs_Agglomerative'
    this_result_path.mkdir(parents=True, exist_ok=True)

    # decompose the data to remove noise and reduce data dimensions
    pca = PCA(n_components=N_COMPONENTS, svd_solver='randomized', random_state=random_state)
    unROI_activation_PCs = pca.fit_transform(unROI_activation.T).T

    # Finding the Pearson distance (1 - r) between voxels of ROI and unROI PCs
    coactivation = pairwise_distances(ROI_activation, unROI_activation_PCs, metric='correlation', n_jobs=64)
    label_dict = {
        n_clusters:
            AgglomerativeClustering(
                n_clusters=n_clusters,
                affinity='euclidean', linkage='ward',
                connectivity=spatial_connectivity
            ).fit_predict(coactivation) + 1
        for n_clusters in [3]
    }

    # plot and save image
    for cluster_n in [3]:
        label = np.zeros_like(ROI_studies_per_voxel, dtype=np.int8)
        label[ROI_voxels_to_use] = label_dict[cluster_n]
        label_img_data = ROI_masker.unmask(label)
        label_img = nib.nifti1.Nifti1Image(label_img_data, header=ROI_IMAGE.header, affine=ROI_IMAGE.affine, )
        nib.save(label_img, str(this_result_path / f'run_{random_state}_K{cluster_n}.nii.gz'))

# Minimum Studies

Finally, we examine the sensitivity of our topography to the initial activation threshold. By varying the `MIN_STUDIES` parameter ($100, 150$, and $200$), we actively alter the signal-to-noise ratio and the total number of eligible VMPFC voxels included in the analysis.

We perform 100 independent K-Means runs for each threshold level. Consistent clustering outputs across these distinct thresholds would strongly indicate that the core functional architecture of the VMPFC is highly resilient to data exclusion criteria.

In [15]:
for min_study in [100, 150, 200]:
    ROI_voxels_to_use = ROI_studies_per_voxel >= min_study
    ROI_activation = ROI_activation_raw[ROI_voxels_to_use, :]
    print(f'ROI dataset: (n voxels, n studies) = {ROI_activation.data.shape}.')

    unROI_voxels_to_use = unROI_studies_per_voxel >= min_study
    unROI_activation = unROI_activation_raw[unROI_voxels_to_use, :]
    print(f'unROI dataset: (n voxels, n studies) = {unROI_activation.data.shape}.')

    this_result_path = RESULTS_PATH / f'robust/runs_ms{min_study}'
    this_result_path.mkdir(exist_ok=True, parents=True)
    for random_state in trange(100, desc=f'min_studies={min_study}'):
        # decompose the data to remove noise and reduce data dimensions
        pca = PCA(n_components=N_COMPONENTS, svd_solver='randomized', random_state=random_state)
        unROI_activation_PCs = pca.fit_transform(unROI_activation.T).T

        # Finding the Pearson distance (1 - r) between voxels of ROI and unROI PCs
        coactivation = pairwise_distances(ROI_activation, unROI_activation_PCs, metric='correlation', n_jobs=64)
        label_dict = {
            n_clusters:
                KMeans(
                    n_clusters=n_clusters,
                    init='random',
                    n_init=1000,
                    random_state=random_state,
                ).fit_predict(coactivation) + 1
            for n_clusters in [3]
        }

        # plot and save image
        for cluster_n in [3]:
            label = np.zeros_like(ROI_studies_per_voxel, dtype=np.int8)
            label[ROI_voxels_to_use] = label_dict[cluster_n]
            label_img_data = ROI_masker.unmask(label)
            label_img = nib.nifti1.Nifti1Image(label_img_data, header=ROI_IMAGE.header, affine=ROI_IMAGE.affine, )
            nib.save(label_img, str(this_result_path / f'run_{random_state}_K{cluster_n}.nii.gz'))


../results/robust/runs_ms100_nc100
../results/robust/runs_ms150_nc100
../results/robust/runs_ms200_nc100
../results/robust/runs_Spectral
../results/robust/runs_Agglomerative
